# Navigating Text in High Dimensions

*Dataharvest 2026 — hands-on notebook*

Keyword search is great when you know what you're looking for. But what about the
structure you *don't* know is there — the topics, the patterns, the outliers?

In this notebook we take a pile of tweets and turn it into a **map you can explore**:

1. Turn text into numbers (**embeddings**)
2. Measure **meaning** with cosine similarity → **semantic search**
3. **Project** to 2D with UMAP and plot it
4. **Find groups** with HDBSCAN — and see what *doesn't* fit
5. **Play** with the knobs and watch the map change

No machine-learning background needed. Run each cell with **Shift+Enter**.
Cells marked **TRY IT** are yours to change and re-run.

Slides: <https://textembeddings.resolve.works>


## 0. Setup

Installs the three libraries we need (~1 min on Colab). Everything else
(pandas, scikit-learn, matplotlib) is already there.

In [ ]:
!pip install -q sentence-transformers umap-learn plotly

In [ ]:
import textwrap
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sentence_transformers import SentenceTransformer, util

pd.set_option("display.max_colwidth", 100)

## 1. Embedding text

An **embedding model** reads a piece of text and turns it into a list of numbers — a
**vector**. We use [`all-MiniLM-L6-v2`](https://huggingface.co/sentence-transformers/all-MiniLM-L6-v2):
small, fast, runs on a laptop with no GPU. The first run downloads it (~80 MB).

In [ ]:
model = SentenceTransformer("sentence-transformers/all-MiniLM-L6-v2")

embedding = model.encode("The sun is shining")

print("type: ", type(embedding).__name__)
print("shape:", embedding.shape, " <- 384 numbers")
print(embedding[:8])

Any text — a word, a sentence, a tweet — becomes **384 numbers**. The text now
lives at a point in a 384-dimensional space.

## 2. Comparing meaning

Texts with similar meaning get similar vectors. We compare them with **cosine
similarity**: `1.0` = same direction (same meaning), `0.0` = unrelated.

Here's the example from the slides.

In [ ]:
query = "The sun is shining"
candidates = [
    "Bring an umbrella",
    "The weather is great",
    "It's daytime",
]

q_emb = model.encode(query)
c_emb = model.encode(candidates)
scores = util.cos_sim(q_emb, c_emb)[0]

for text, score in sorted(zip(candidates, scores), key=lambda x: -x[1]):
    print(f"{score:.3f}   {text}")

No shared keywords needed — *"It's daytime"* wins because it's closest in **meaning**.

> **TRY IT** — change the `query` and `candidates` and re-run.

## 3. Load some real data

Now a real corpus: tweets from the
[TweetTopic](https://huggingface.co/datasets/cardiffnlp/tweet_topic_single) dataset,
where humans labelled each tweet with one of **6 topics**. We keep those labels aside
as a "ground truth" to compare against later.

We fetch it straight from Hugging Face (no account needed) and tidy it up. The dataset
hides URLs and anonymous usernames behind placeholders; we **drop those** (they're
noise) but **keep the names of public figures** (they carry topic meaning — think
*Greta Thunberg* or a football player).

In [ ]:
import io, re, urllib.request

URL = ("https://huggingface.co/datasets/cardiffnlp/tweet_topic_single/"
       "resolve/main/dataset/split_temporal/train_2020.single.json")

req = urllib.request.Request(URL, headers={"User-Agent": "Mozilla/5.0"})
raw = urllib.request.urlopen(req).read().decode("utf-8")
df = pd.read_json(io.StringIO(raw), lines=True)

def clean(text):
    text = text.replace("{{URL}}", "")          # drop links
    text = text.replace("{{USERNAME}}", "")     # drop anonymous @handles
    text = re.sub(r"\{@(.*?)@\}", r"\1", text)  # {@Greta Thunberg@} -> Greta Thunberg
    return re.sub(r"\s+", " ", text).strip()

print("BEFORE:", df["text"].iloc[6])
df["text"] = df["text"].map(clean)
df = df[df["text"].str.len() >= 15]             # drop near-empty tweets
print("AFTER: ", df["text"].iloc[6])

The topics are very unbalanced (lots of sport, little arts). For a clean demo we
take an **even sample** — up to 100 tweets per topic — and shuffle.

In [ ]:
parts = []
for topic, group in df.groupby("label_name"):
    parts.append(group.sample(min(len(group), 100), random_state=42))

df = pd.concat(parts).sample(frac=1, random_state=42).reset_index(drop=True)

print(f"{len(df)} tweets")
print(df["label_name"].value_counts())
df[["text", "label_name"]].head()

## 4. Embed the whole corpus

One call embeds every tweet. A few hundred short texts take a few seconds.

In [ ]:
texts = df["text"].tolist()
embeddings = model.encode(texts, show_progress_bar=True)

print("embeddings shape:", embeddings.shape, " <- (tweets, 384)")

## 5. Semantic search

Search by **meaning** instead of keywords: embed a query, find the tweets whose
vectors point in the most similar direction.

In [ ]:
def search(query, k=5):
    q_emb = model.encode(query)
    scores = util.cos_sim(q_emb, embeddings)[0]
    for i in scores.argsort(descending=True)[:k]:
        i = int(i)
        print(f"{scores[i]:.3f}   {df['text'].iloc[i]}")

search("a thrilling football match")

> **TRY IT** — change the query. Try something with no obvious keywords, e.g.
`"new gadget launch"`, `"feeling stressed"`, or `"protest against the government"`.

## 6. Draw the map (UMAP → 2D)

Each tweet is a point in **384 dimensions** — impossible to picture. **UMAP** squeezes
that to **2D** while keeping neighbours close to neighbours, so we can plot it.

We colour each point by its **human topic** and you can **hover to read the tweet**.

In [ ]:
from umap import UMAP

viz = UMAP(
    n_neighbors=15,    # neighbourhood size: local (small) <-> global (large)
    min_dist=0.1,      # how tightly points may pack for the picture
    n_components=2,
    metric="cosine",
    random_state=42,
)
coords = viz.fit_transform(embeddings)
df["x"], df["y"] = coords[:, 0], coords[:, 1]
df["hover"] = df["text"].map(lambda t: "<br>".join(textwrap.wrap(t, 60)))
print("projected to 2D")

In [ ]:
import plotly.express as px

def show_map(color, title, cmap=None, order=None):
    fig = px.scatter(
        df, x="x", y="y", color=color, custom_data=["hover"],
        title=title, width=900, height=600, opacity=0.8,
        color_discrete_map=cmap, category_orders=order,
    )
    fig.update_traces(hovertemplate="%{customdata[0]}<extra></extra>")
    fig.update_layout(legend_title_text="", xaxis_title="", yaxis_title="")
    fig.show()

show_map("label_name", "Tweets in 2D — colour = human topic")

Tweets about the same topic land near each other — **without anyone telling UMAP
the labels**. It only saw the 384-number vectors.

## 7. Find groups automatically (HDBSCAN)

What if we *didn't* have the human labels? **HDBSCAN** finds dense groups on its own,
and marks points in sparse areas as **noise** (`-1`) — *"doesn't fit any group"*.

**One important step first.** Density-based clustering struggles in hundreds of
dimensions (everything looks equally far apart). The standard fix
([UMAP docs](https://umap-learn.readthedocs.io/en/latest/clustering.html),
[BERTopic](https://maartengr.github.io/BERTopic/getting_started/dim_reduction/dim_reduction.html))
is to first reduce to a **handful of dimensions** — not 2 (too crushed) and not 384
(too sparse), but something in between like **5**. We use `min_dist=0` here to let
clusters pack tightly.

So: **reduce to 5D for the algorithm**, but keep the **2D map for our eyes**.

In [ ]:
from sklearn.cluster import HDBSCAN

# 5D UMAP just for clustering (separate from the 2D map above)
cluster_space = UMAP(
    n_neighbors=15, min_dist=0.0, n_components=5,
    metric="cosine", random_state=42,
).fit_transform(embeddings)

df["cluster"] = HDBSCAN(min_cluster_size=10).fit_predict(cluster_space)

n_clusters = df.loc[df["cluster"] != -1, "cluster"].nunique()
n_noise = int((df["cluster"] == -1).sum())
print(f"found {n_clusters} clusters")
print(f"{n_noise} tweets left as noise (fit no group)")

Now colour the **same 2D map** by the groups HDBSCAN discovered (grey = noise).

In [ ]:
df["group"] = df["cluster"].map(lambda c: "noise" if c == -1 else f"cluster {c}")
order = {"group": ["noise"] + [f"cluster {c}" for c in sorted(df.loc[df['cluster'] != -1, 'cluster'].unique())]}

show_map("group", "Same map — colour = discovered cluster",
         cmap={"noise": "lightgray"}, order=order)

## 8. What is each cluster about?

A cluster is just a bag of tweets. To understand it we pull out the **words that are
distinctive** to each one (TF-IDF), plus a couple of example tweets.

In [ ]:
from sklearn.feature_extraction.text import TfidfVectorizer, ENGLISH_STOP_WORDS

clusters = sorted(df.loc[df["cluster"] != -1, "cluster"].unique())
docs = [" ".join(df.loc[df["cluster"] == c, "text"]) for c in clusters]

stop = list(ENGLISH_STOP_WORDS) + ["user", "amp", "rt", "http", "https", "com"]
vec = TfidfVectorizer(stop_words=stop, token_pattern=r"[A-Za-z]{3,}", max_features=3000)
tfidf = vec.fit_transform(docs)
terms = vec.get_feature_names_out()

for row, c in sorted(zip(tfidf, clusters), key=lambda z: -(df["cluster"] == z[1]).sum()):
    keywords = [terms[j] for j in row.toarray().ravel().argsort()[::-1][:8]]
    members = df.loc[df["cluster"] == c]
    print(f"CLUSTER {c}  ({len(members)} tweets)")
    print("  keywords:", ", ".join(keywords))
    for t in members["text"].head(2):
        print("   •", t[:100])
    print()

## 9. The outliers

The tweets left as **noise** didn't fit any group.

In [ ]:
for t in df.loc[df["cluster"] == -1, "text"].head(10):
    print("•", t)

## 10. How your choices shape the map

Every result above depended on choices we made. Let's make that **visible**. Below is
the whole clustering pipeline in one function, so we can sweep a setting and watch the
map change.

In [ ]:
def run_clusters(n_neighbors=15, n_components=5, min_cluster_size=10):
    reduced = UMAP(n_neighbors=n_neighbors, min_dist=0.0, n_components=n_components,
                   metric="cosine", random_state=42).fit_transform(embeddings)
    return HDBSCAN(min_cluster_size=min_cluster_size).fit_predict(reduced)

def plot_map(ax, labels, title):
    cats = sorted(set(labels), key=lambda c: (c == -1, c))
    for i, c in enumerate(cats):
        m = labels == c
        color = "lightgray" if c == -1 else plt.cm.tab20(i % 20)
        ax.scatter(coords[m, 0], coords[m, 1], s=10, alpha=0.8, color=color)
    n_c = sum(1 for c in cats if c != -1)
    ax.set_title(f"{title}\n{n_c} clusters, {int((labels==-1).sum())} noise", fontsize=11)
    ax.set_xticks([]); ax.set_yticks([])

**`min_cluster_size`** — how big must a group be to count? Small = many tiny
clusters; large = a few broad ones, then everything collapses.

In [ ]:
fig, axes = plt.subplots(1, 4, figsize=(20, 5))
for ax, mcs in zip(axes, [5, 10, 20, 40]):
    plot_map(ax, run_clusters(min_cluster_size=mcs), f"min_cluster_size = {mcs}")
plt.tight_layout(); plt.show()

**`n_components`** — the dimensions we cluster in. This is the choice from step 7:
**2** crushes everything into one blob, **5** separates nicely, very high gets sparse
and noisy. (That's why we don't cluster on the 2D picture directly.)

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(16, 5))
for ax, nc in zip(axes, [2, 5, 15]):
    plot_map(ax, run_clusters(n_components=nc), f"n_components = {nc}")
plt.tight_layout(); plt.show()

> **TRY IT** — change the numbers in the sweeps above (or call
`run_clusters(n_neighbors=30)` yourself). There is **no single right answer** — the
point is to find a view that's useful for *your* question.

## That's the workflow

From a pile of text to an explorable map, with clusters and outliers — and you never
told the machine what to look for.

**Use it on your own data:** replace step 3 with any column of text — articles,
comments, FOI responses, transcripts — and run the rest.

**Keep exploring:**
- [sbert.net](https://sbert.net) — the embeddings library
- [MTEB leaderboard](https://huggingface.co/spaces/mteb/leaderboard) — compare embedding models
- [UMAP docs](https://umap-learn.readthedocs.io/) · [TensorFlow Projector](https://projector.tensorflow.org/)
- Documents → text: [docling](https://github.com/docling-project/docling), [markitdown](https://github.com/microsoft/markitdown)
